# Interpolating Models Notebook
This notebook interpolates two models by paramaterizing a line between each parameter. (TODO Fix that sentence). 

In [11]:
import sys
from pathlib import Path
cwd = Path.cwd()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'REPO_GUIDE.md').exists()), cwd)
sys.path.insert(0, str(ROOT))

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import copy


import data
import models

In [3]:
def get_model(model_path):
    snapshot = torch.load(model_path, map_location="cpu", weights_only=False)
    state_dict = snapshot['state_dict']

    # These checkpoints were trained with BatchNorm disabled.
    model = models.ResNet(m_type='resnet18', num_classes=10, BatchNorm=False)
    model.load_state_dict(state_dict)
    return model

In [4]:
# Set these inputs for each interactive notebook run.
model1_path = Path(input('Model 1 checkpoint: '))
model2_path = Path(input('Model 2 checkpoint: '))
DATASET_ROOT = Path(input(f"Dataset root [{ROOT / '_CIFAR'}]: " ) or ROOT / '_CIFAR')
NUM_POINTS = int(input('Number of interpolation points [25]: ') or 25)
BATCH_SIZE = int(input('Evaluation batch size [2048]: ') or 2048)

model1 = get_model(model1_path)
model2 = get_model(model2_path)

In [14]:
def interpolate_models(model1, model2):
    def model_at_t(t):
        model = copy.deepcopy(model1)

        with torch.no_grad():
            for p, p1, p2 in zip(model.parameters(), model1.parameters(), model2.parameters()):
                p.copy_(t * p1 + (1 - t) * p2)
        return model
    return model_at_t

In [6]:
def plot_data(model1, model2, data_loader, loss_func):
    with torch.no_grad():
        model_func = interpolate_models(model1, model2)

        inputs = np.linspace(0, 1, NUM_POINTS)
        losses = []
        for t in inputs:
            model = model_func(t)
            model.eval()
            loss = 0
            i = 0
            for images, targets in data_loader:
                i += 1
                outputs = model(images)
                loss += loss_func(outputs, targets)
            losses.append(loss / i)


        plt.plot(inputs, losses)
        plt.xlabel('t')
        plt.ylabel('Loss')
        plt.title('Parametrization from Model 1 to Model 2')
        plt.show()


In [15]:
im_size = (32, 32)
num_classes = 10
mean = [0.4914, 0.4822, 0.4465]
std = [0.2470, 0.2435, 0.2616]

transform =  transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=mean, std=std)]) if im_size[0] == 32 else transforms.Compose(
        [transforms.Resize(im_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)]
        )

dataset = datasets.CIFAR10(DATASET_ROOT, train=True, download=False, transform=transform)

data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

plot_data(model1, model2, data_loader, nn.CrossEntropyLoss())

KeyboardInterrupt: 

TODO: 
* Load a datapoint and test plot_data
* Add Comments
* Fix description sentences
* Make graph nice
* run tests that save lots of checkpoints